# Анализ жизненного цикла ФП/СФП: сценарий и ИПУ

Исследуем, как проходят этапы урегулирования факторов проблемности:
- Какие исходы сценариев встречаются в разрезе по сегментам
- Какая доля ФП переходит из сценария в ИПУ
- Какие исходы ИПУ и их распределение
- Временные характеристики: длительность сценариев и ИПУ

**Источник:** `crm_last_version.csv`

## 0. Импорт и конфигурация

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

plt.rcParams.update({"font.family": "sans-serif",
                      "font.sans-serif": ["Arial", "DejaVu Sans"],
                      "axes.unicode_minus": False})

# ── Пути ──
DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"

# ── Период ──
DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO   = pd.Timestamp("2025-12-31")

# ── Маппинг сегментов ──
SEGMENT_MAP = {
    "ДМСБ (микро)":   "МкБ",
    "ДМСБ (малый)":   "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ":            "ККБ",
}
SEG_ORDER = ["МкБ", "МСБ", "ККБ"]

ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

# Ключевые колонки для анализа
KEY_COLS = [
    "X_INN", "NUMBER_FP_SFP", "TYPE_FP", "X_AREA_RESP",
    "IDENTIFICATION_DATE", "VAL",
    "DESC_TEXT", "DESC_TEXT_1", "SCRIPT",
    "END_DATE_SCR_PLAN", "END_DATE_SCR_FCT",
    "FIRST_END_DATE_EVENT", "END_EVENT_DATE_FACT",
    "DEFOLT",
]


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


print("Конфигурация задана.")

In [ ]:
# ── Загрузка CRM ──
df = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
df.columns = df.columns.str.strip()
print(f"Загружено: {len(df):,} строк × {df.shape[1]} колонок")

# Фильтр по источникам
if "VAL" in df.columns:
    df = df[df["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()
    print(f"После фильтра по источникам: {len(df):,}")

# ИНН
df["inn"] = df["X_INN"].apply(normalize_inn)

# Даты
df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()
print(f"После фильтра по периоду: {len(df):,}")

# Сегмент
df["segment"] = df["X_AREA_RESP"].str.strip().map(SEGMENT_MAP)
unmapped = df["segment"].isna().sum()
if unmapped > 0:
    print(f"Без сегмента: {unmapped:,} (удалены)")
    df = df[df["segment"].notna()].copy()

# Номер ФП и тип
df["fp_num"] = df["NUMBER_FP_SFP"].str.strip()
df["fp_type"] = df["TYPE_FP"].str.strip().replace({"FP": "ФП", "SFP": "СФП"})

# Дедупликация
before = len(df)
df = df.drop_duplicates(subset=["inn", "fp_num", "fp_type", "IDENTIFICATION_DATE"]).copy()
print(f"После дедупликации: {len(df):,} (удалено {before - len(df):,} дубликатов)")

# Парсинг дат сценария и ИПУ
for col in ["END_DATE_SCR_PLAN", "END_DATE_SCR_FCT",
            "FIRST_END_DATE_EVENT", "END_EVENT_DATE_FACT"]:
    df[col + "_dt"] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")

# Нормализуем текстовые поля результатов
df["result_scenario"] = df["DESC_TEXT"].fillna("").str.strip()
df["result_ipu"] = df["DESC_TEXT_1"].fillna("").str.strip()
df["script"] = df["SCRIPT"].fillna("").str.strip()

# Признак: перешёл в ИПУ
df["has_ipu"] = (
    (df["result_ipu"] != "") |
    df["FIRST_END_DATE_EVENT_dt"].notna() |
    df["END_EVENT_DATE_FACT_dt"].notna()
)

# Признак: сценарий завершён
df["has_scenario_result"] = df["result_scenario"] != ""

print(f"\nИтого: {len(df):,} записей, {df['inn'].nunique():,} уникальных ИНН")
print(f"Сегменты: {df['segment'].value_counts().reindex(SEG_ORDER).to_dict()}")

## 1. Обзор данных

Общая статистика и заполняемость ключевых колонок.

In [ ]:
total = len(df)
print(f"Записей: {total:,}")
print(f"Уникальных ИНН: {df['inn'].nunique():,}")
print(f"Уникальных номеров ФП/СФП: {df['fp_num'].nunique():,}")

# Распределение по сегментам
seg_counts = df["segment"].value_counts().reindex(SEG_ORDER)
print(f"\nРаспределение по сегментам:")
for seg, cnt in seg_counts.items():
    print(f"  {seg}: {cnt:,} ({cnt/total*100:.1f}%)")

# Заполняемость ключевых колонок
check_cols = {
    "DESC_TEXT (результат сценария)": "result_scenario",
    "DESC_TEXT_1 (результат ИПУ)": "result_ipu",
    "SCRIPT (сценарий)": "script",
    "END_DATE_SCR_PLAN (план дата сценария)": "END_DATE_SCR_PLAN_dt",
    "END_DATE_SCR_FCT (факт дата сценария)": "END_DATE_SCR_FCT_dt",
    "FIRST_END_DATE_EVENT (план дата ИПУ)": "FIRST_END_DATE_EVENT_dt",
    "END_EVENT_DATE_FACT (факт дата ИПУ)": "END_EVENT_DATE_FACT_dt",
}

print(f"\nЗаполняемость:")
fill_data = []
for label, col in check_cols.items():
    if col.endswith("_dt"):
        filled = df[col].notna().sum()
    else:
        filled = (df[col] != "").sum()
    pct = filled / total * 100
    fill_data.append((label, f"{filled:,}", f"{pct:.1f}%"))

fill_df = pd.DataFrame(fill_data, columns=["Колонка", "Заполнено", "%"])
display(fill_df.style.hide(axis="index"))

## 2. Исходы сценариев

Какие результаты урегулирования по сценарию встречаются
и как они распределены по сегментам.

In [ ]:
# Общие исходы сценария
sc = df["result_scenario"].copy()
sc[sc == ""] = "[пусто]"

vc = sc.value_counts()
print(f"Уникальных исходов сценария: {len(vc):,}\n")
print("Все варианты:")
for val, cnt in vc.items():
    print(f"  {cnt:>8,} ({cnt/total*100:5.1f}%)  {val}")

In [ ]:
# Кросс-таблица: сегмент × результат сценария
df["_sc"] = sc  # временная колонка

cross_sc = pd.crosstab(df["_sc"], df["segment"], margins=True, margins_name="Итого")
# Упорядочиваем колонки
col_order = [s for s in SEG_ORDER if s in cross_sc.columns] + ["Итого"]
cross_sc = cross_sc[col_order].sort_values("Итого", ascending=False)

print("Абсолютные значения:")
display(cross_sc)

# Процентная таблица (по столбцу = внутри сегмента)
cross_sc_pct = pd.crosstab(df["_sc"], df["segment"], normalize="columns") * 100
cross_sc_pct = cross_sc_pct[[s for s in SEG_ORDER if s in cross_sc_pct.columns]]
cross_sc_pct = cross_sc_pct.loc[cross_sc.index[cross_sc.index != "Итого"]]

print("\nПроцент внутри сегмента:")
display(cross_sc_pct.round(1).style.background_gradient(cmap="YlOrRd", axis=None))

df.drop(columns=["_sc"], inplace=True)

In [ ]:
# Топ-20 номеров ФП для каждого исхода сценария
sc_outcomes = df[df["result_scenario"] != ""]["result_scenario"].unique()

for outcome in sorted(sc_outcomes):
    subset = df[df["result_scenario"] == outcome]
    top = subset.groupby(["fp_num", "fp_type"]).size().sort_values(ascending=False).head(20)
    print(f"\n{'='*60}")
    print(f"Исход: {outcome} ({len(subset):,} записей)")
    print(f"{'='*60}")
    for (fp, tp), cnt in top.items():
        print(f"  {cnt:>6,}  {fp} ({tp})")

## 3. Переход из сценария в ИПУ

Определяем, какая доля ФП переходит на этап ИПУ,
и при каких исходах сценария это происходит.

In [ ]:
# ── Общая воронка ──
n_total = len(df)
n_with_scenario = df["has_scenario_result"].sum()
n_with_ipu = df["has_ipu"].sum()

print("Воронка жизненного цикла ФП:")
print(f"  Всего ФП/СФП:               {n_total:>8,} (100%)")
print(f"  С результатом сценария:      {n_with_scenario:>8,} ({n_with_scenario/n_total*100:.1f}%)")
print(f"  Перешли в ИПУ:               {n_with_ipu:>8,} ({n_with_ipu/n_total*100:.1f}%)")
if n_with_scenario > 0:
    print(f"  % перехода (от сценария):    {n_with_ipu/n_with_scenario*100:.1f}%")

In [ ]:
# ── Воронка по сегментам ──
funnel_data = []
for seg in SEG_ORDER:
    s = df[df["segment"] == seg]
    n = len(s)
    n_sc = s["has_scenario_result"].sum()
    n_ipu = s["has_ipu"].sum()
    funnel_data.append((
        seg, f"{n:,}",
        f"{n_sc:,} ({n_sc/n*100:.1f}%)",
        f"{n_ipu:,} ({n_ipu/n*100:.1f}%)",
        f"{n_ipu/max(n_sc,1)*100:.1f}%"
    ))

funnel_df = pd.DataFrame(funnel_data,
    columns=["Сегмент", "Всего ФП", "С результатом сценария",
             "Перешли в ИПУ", "% перехода"])
display(funnel_df.style.hide(axis="index"))

In [ ]:
# ── При каких исходах сценария назначается ИПУ ──
# Кросс-таблица: исход сценария × перешёл в ИПУ
df_with_sc = df[df["has_scenario_result"]].copy()

cross_ipu = df_with_sc.groupby("result_scenario").agg(
    Всего=("has_ipu", "size"),
    В_ИПУ=("has_ipu", "sum"),
)
cross_ipu["Без_ИПУ"] = cross_ipu["Всего"] - cross_ipu["В_ИПУ"]
cross_ipu["% в ИПУ"] = (cross_ipu["В_ИПУ"] / cross_ipu["Всего"] * 100).round(1)
cross_ipu = cross_ipu.sort_values("Всего", ascending=False)

print("Переход в ИПУ по исходу сценария:")
display(
    cross_ipu.style
    .background_gradient(subset=["% в ИПУ"], cmap="YlOrRd")
    .format({"Всего": "{:,}", "В_ИПУ": "{:,}", "Без_ИПУ": "{:,}"})
)

In [ ]:
# ── Переход в ИПУ по номерам ФП (по сегментам) ──
for seg in SEG_ORDER:
    s = df[df["segment"] == seg]
    fp_ipu = s.groupby(["fp_num", "fp_type"]).agg(
        Всего=("has_ipu", "size"),
        В_ИПУ=("has_ipu", "sum"),
    )
    fp_ipu["% в ИПУ"] = (fp_ipu["В_ИПУ"] / fp_ipu["Всего"] * 100).round(1)
    fp_ipu = fp_ipu.sort_values("% в ИПУ", ascending=False)

    print(f"\n{'='*60}")
    print(f"{seg}: переход в ИПУ по номерам ФП (топ-30)")
    print(f"{'='*60}")
    display(
        fp_ipu.head(30).style
        .background_gradient(subset=["% в ИПУ"], cmap="YlOrRd")
        .format({"Всего": "{:,}", "В_ИПУ": "{:,}"})
    )

## 4. Исходы ИПУ

Результаты урегулирования на этапе ИПУ
и их распределение по сегментам.

In [ ]:
# Общие исходы ИПУ
ipu_data = df[df["has_ipu"]].copy()
print(f"Записей с ИПУ: {len(ipu_data):,}\n")

ipu_res = ipu_data["result_ipu"].copy()
ipu_res[ipu_res == ""] = "[пусто — только даты ИПУ]"

vc_ipu = ipu_res.value_counts()
print("Все варианты исходов ИПУ:")
for val, cnt in vc_ipu.items():
    print(f"  {cnt:>8,} ({cnt/len(ipu_data)*100:5.1f}%)  {val}")

In [ ]:
# Кросс-таблица: сегмент × результат ИПУ
ipu_data["_ipu"] = ipu_res.values

cross_ipu_seg = pd.crosstab(ipu_data["_ipu"], ipu_data["segment"],
                             margins=True, margins_name="Итого")
col_order = [s for s in SEG_ORDER if s in cross_ipu_seg.columns] + ["Итого"]
cross_ipu_seg = cross_ipu_seg[col_order].sort_values("Итого", ascending=False)

print("Абсолютные значения:")
display(cross_ipu_seg)

cross_ipu_pct = pd.crosstab(ipu_data["_ipu"], ipu_data["segment"],
                              normalize="columns") * 100
cross_ipu_pct = cross_ipu_pct[[s for s in SEG_ORDER if s in cross_ipu_pct.columns]]
cross_ipu_pct = cross_ipu_pct.loc[cross_ipu_seg.index[cross_ipu_seg.index != "Итого"]]

print("\nПроцент внутри сегмента:")
display(cross_ipu_pct.round(1).style.background_gradient(cmap="YlOrRd", axis=None))

ipu_data.drop(columns=["_ipu"], inplace=True)

In [ ]:
# Топ-20 номеров ФП для каждого исхода ИПУ
ipu_outcomes = ipu_data[ipu_data["result_ipu"] != ""]["result_ipu"].unique()

for outcome in sorted(ipu_outcomes):
    subset = ipu_data[ipu_data["result_ipu"] == outcome]
    top = subset.groupby(["fp_num", "fp_type"]).size().sort_values(ascending=False).head(20)
    print(f"\n{'='*60}")
    print(f"Исход ИПУ: {outcome} ({len(subset):,} записей)")
    print(f"{'='*60}")
    for (fp, tp), cnt in top.items():
        print(f"  {cnt:>6,}  {fp} ({tp})")

In [ ]:
# ── Сквозная воронка: исход сценария → ИПУ → исход ИПУ ──
df_through = df[df["has_scenario_result"] & df["has_ipu"]].copy()
df_through["_ipu_res"] = df_through["result_ipu"].copy()
df_through.loc[df_through["_ipu_res"] == "", "_ipu_res"] = "[ИПУ без результата]"

through = pd.crosstab(df_through["result_scenario"], df_through["_ipu_res"],
                       margins=True, margins_name="Итого")
through = through.sort_values("Итого", ascending=False)

print("Сквозная воронка: исход сценария → исход ИПУ")
display(through)

df_through.drop(columns=["_ipu_res"], inplace=True)

## 5. Временной анализ

Длительность этапов сценария и ИПУ, соблюдение плановых сроков.

In [ ]:
# ── Длительность сценария: факт дата окончания − дата выявления ──
df["scr_duration"] = (df["END_DATE_SCR_FCT_dt"] - df["dt"]).dt.days

# Фильтруем разумные значения (0..3000 дней)
scr_valid = df[(df["scr_duration"] >= 0) & (df["scr_duration"] <= 3000)]

print("Длительность сценария (дни): дата окончания факт − дата выявления")
print(f"Записей с данными: {len(scr_valid):,}\n")

scr_stats = []
for seg in SEG_ORDER:
    s = scr_valid[scr_valid["segment"] == seg]["scr_duration"]
    if len(s) > 0:
        scr_stats.append((seg, f"{len(s):,}", f"{s.mean():.0f}",
                          f"{s.median():.0f}", f"{s.min():.0f}", f"{s.max():.0f}"))

s_all = scr_valid["scr_duration"]
scr_stats.append(("ВСЕ", f"{len(s_all):,}", f"{s_all.mean():.0f}",
                  f"{s_all.median():.0f}", f"{s_all.min():.0f}", f"{s_all.max():.0f}"))

scr_df = pd.DataFrame(scr_stats,
    columns=["Сегмент", "N", "Среднее", "Медиана", "Мин", "Макс"])
display(scr_df.style.hide(axis="index"))

# Гистограмма
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, seg in zip(axes, SEG_ORDER):
    data = scr_valid[scr_valid["segment"] == seg]["scr_duration"]
    if len(data) > 0:
        ax.hist(data, bins=50, edgecolor="white", alpha=0.8)
    ax.set_title(f"{seg} (N={len(data):,})")
    ax.set_xlabel("Дни")
fig.suptitle("Длительность сценария (дни)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Длительность ИПУ: факт дата окончания ИПУ − факт дата окончания сценария ──
df["ipu_duration"] = (df["END_EVENT_DATE_FACT_dt"] - df["END_DATE_SCR_FCT_dt"]).dt.days

ipu_valid = df[(df["ipu_duration"] >= 0) & (df["ipu_duration"] <= 3000)]

print("Длительность ИПУ (дни): факт дата ИПУ − факт дата сценария")
print(f"Записей с данными: {len(ipu_valid):,}\n")

ipu_stats = []
for seg in SEG_ORDER:
    s = ipu_valid[ipu_valid["segment"] == seg]["ipu_duration"]
    if len(s) > 0:
        ipu_stats.append((seg, f"{len(s):,}", f"{s.mean():.0f}",
                          f"{s.median():.0f}", f"{s.min():.0f}", f"{s.max():.0f}"))

i_all = ipu_valid["ipu_duration"]
if len(i_all) > 0:
    ipu_stats.append(("ВСЕ", f"{len(i_all):,}", f"{i_all.mean():.0f}",
                      f"{i_all.median():.0f}", f"{i_all.min():.0f}", f"{i_all.max():.0f}"))

ipu_dur_df = pd.DataFrame(ipu_stats,
    columns=["Сегмент", "N", "Среднее", "Медиана", "Мин", "Макс"])
display(ipu_dur_df.style.hide(axis="index"))

# Гистограмма
if len(ipu_valid) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, seg in zip(axes, SEG_ORDER):
        data = ipu_valid[ipu_valid["segment"] == seg]["ipu_duration"]
        if len(data) > 0:
            ax.hist(data, bins=50, edgecolor="white", alpha=0.8, color="coral")
        ax.set_title(f"{seg} (N={len(data):,})")
        ax.set_xlabel("Дни")
    fig.suptitle("Длительность ИПУ (дни)", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Соблюдение сроков: план vs факт ──

# Сценарий: просрочка = факт > план
scr_both = df[df["END_DATE_SCR_FCT_dt"].notna() & df["END_DATE_SCR_PLAN_dt"].notna()].copy()
scr_both["scr_delay"] = (scr_both["END_DATE_SCR_FCT_dt"] - scr_both["END_DATE_SCR_PLAN_dt"]).dt.days
scr_both["scr_late"] = scr_both["scr_delay"] > 0

print("Соблюдение сроков сценария (план vs факт):")
print(f"  Записей с обеими датами: {len(scr_both):,}")
for seg in SEG_ORDER:
    s = scr_both[scr_both["segment"] == seg]
    late = s["scr_late"].sum()
    print(f"  {seg}: просрочено {late:,} из {len(s):,} ({late/max(len(s),1)*100:.1f}%), "
          f"средняя задержка {s.loc[s['scr_late'], 'scr_delay'].mean():.0f} дн." if late > 0 else
          f"  {seg}: просрочено 0")

# ИПУ: просрочка = факт > план
ipu_both = df[df["END_EVENT_DATE_FACT_dt"].notna() & df["FIRST_END_DATE_EVENT_dt"].notna()].copy()
if len(ipu_both) > 0:
    ipu_both["ipu_delay"] = (ipu_both["END_EVENT_DATE_FACT_dt"] - ipu_both["FIRST_END_DATE_EVENT_dt"]).dt.days
    ipu_both["ipu_late"] = ipu_both["ipu_delay"] > 0

    print(f"\nСоблюдение сроков ИПУ (план vs факт):")
    print(f"  Записей с обеими датами: {len(ipu_both):,}")
    for seg in SEG_ORDER:
        s = ipu_both[ipu_both["segment"] == seg]
        late = s["ipu_late"].sum()
        if late > 0:
            print(f"  {seg}: просрочено {late:,} из {len(s):,} ({late/len(s)*100:.1f}%), "
                  f"средняя задержка {s.loc[s['ipu_late'], 'ipu_delay'].mean():.0f} дн.")
        else:
            print(f"  {seg}: просрочено 0 из {len(s):,}")
else:
    print("\nНедостаточно данных для анализа сроков ИПУ.")

## 6. Динамика по времени

Помесячная динамика завершения сценариев и перехода в ИПУ.

In [ ]:
# Помесячная динамика по дате выявления
df["month"] = df["dt"].dt.to_period("M")

monthly = df.groupby("month").agg(
    Всего=("has_ipu", "size"),
    Сценарий_завершён=("has_scenario_result", "sum"),
    В_ИПУ=("has_ipu", "sum"),
)
monthly["% в ИПУ"] = (monthly["В_ИПУ"] / monthly["Всего"] * 100).round(1)
monthly.index = monthly.index.astype(str)

print("Помесячная динамика:")
display(monthly)

# График тренда % перехода в ИПУ
fig, ax1 = plt.subplots(figsize=(14, 5))
x = range(len(monthly))
ax1.bar(x, monthly["Всего"], alpha=0.3, label="Всего ФП", color="steelblue")
ax1.bar(x, monthly["В_ИПУ"], alpha=0.7, label="Перешли в ИПУ", color="coral")
ax1.set_ylabel("Количество")
ax1.set_xticks(x)
ax1.set_xticklabels(monthly.index, rotation=45, ha="right", fontsize=8)
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(x, monthly["% в ИПУ"], color="darkred", marker="o", markersize=4,
         linewidth=2, label="% в ИПУ")
ax2.set_ylabel("% перехода в ИПУ")
ax2.legend(loc="upper right")

plt.title("Динамика перехода ФП в ИПУ по месяцам")
plt.tight_layout()
plt.show()

In [ ]:
# Распределение исходов сценария по кварталам
df["quarter"] = df["dt"].dt.to_period("Q")

sc_q = df[df["has_scenario_result"]].copy()
cross_q = pd.crosstab(sc_q["quarter"].astype(str), sc_q["result_scenario"],
                       normalize="index") * 100

print("Распределение исходов сценария по кварталам (% от итога квартала):")
display(cross_q.round(1).style.background_gradient(cmap="YlOrRd", axis=None))

## 7. Дополнительные проверки

Связь типа сценария с переходом в ИПУ,
повторные ФП и связь с дефолтом.

In [ ]:
# ── Связь SCRIPT (тип сценария) и перехода в ИПУ ──
if df["script"].ne("").sum() > 0:
    script_ipu = df[df["script"] != ""].groupby("script").agg(
        Всего=("has_ipu", "size"),
        В_ИПУ=("has_ipu", "sum"),
    )
    script_ipu["% в ИПУ"] = (script_ipu["В_ИПУ"] / script_ipu["Всего"] * 100).round(1)
    script_ipu = script_ipu.sort_values("Всего", ascending=False)

    print("Тип сценария → переход в ИПУ:")
    display(
        script_ipu.style
        .background_gradient(subset=["% в ИПУ"], cmap="YlOrRd")
        .format({"Всего": "{:,}", "В_ИПУ": "{:,}"})
    )
else:
    print("Колонка SCRIPT не заполнена.")

In [ ]:
# ── Повторные ФП: компании с >1 ФП — чаще ли ИПУ? ──
company_fp = df.groupby("inn").agg(
    n_fp=("fp_num", "size"),
    n_ipu=("has_ipu", "sum"),
)
company_fp["has_any_ipu"] = company_fp["n_ipu"] > 0
company_fp["fp_group"] = pd.cut(company_fp["n_fp"],
    bins=[0, 1, 3, 5, 10, 1000],
    labels=["1 ФП", "2-3 ФП", "4-5 ФП", "6-10 ФП", ">10 ФП"])

repeat_stats = company_fp.groupby("fp_group", observed=True).agg(
    Компаний=("n_fp", "size"),
    С_ИПУ=("has_any_ipu", "sum"),
)
repeat_stats["% с ИПУ"] = (repeat_stats["С_ИПУ"] / repeat_stats["Компаний"] * 100).round(1)

print("Связь количества ФП у компании с наличием ИПУ:")
display(
    repeat_stats.style
    .background_gradient(subset=["% с ИПУ"], cmap="YlOrRd")
    .format({"Компаний": "{:,}", "С_ИПУ": "{:,}"})
)

In [ ]:
# ── Связь с дефолтом ──
if "DEFOLT" in df.columns:
    df["is_default"] = df["DEFOLT"].str.strip().str.upper() == "Y"

    def_stats = df.groupby("is_default").agg(
        Записей=("has_ipu", "size"),
        С_ИПУ=("has_ipu", "sum"),
    )
    def_stats.index = def_stats.index.map({True: "Дефолт (Y)", False: "Не дефолт (N)"})
    def_stats["% с ИПУ"] = (def_stats["С_ИПУ"] / def_stats["Записей"] * 100).round(1)

    print("Связь дефолта с наличием ИПУ:")
    display(
        def_stats.style
        .background_gradient(subset=["% с ИПУ"], cmap="YlOrRd")
        .format({"Записей": "{:,}", "С_ИПУ": "{:,}"})
    )

    # По сегментам
    print("\nПо сегментам:")
    for seg in SEG_ORDER:
        s = df[df["segment"] == seg]
        d_yes = s[s["is_default"]]
        d_no = s[~s["is_default"]]
        ipu_yes = d_yes["has_ipu"].sum()
        ipu_no = d_no["has_ipu"].sum()
        print(f"  {seg}: дефолтные {ipu_yes}/{len(d_yes)} ({ipu_yes/max(len(d_yes),1)*100:.1f}% ИПУ), "
              f"недефолтные {ipu_no}/{len(d_no)} ({ipu_no/max(len(d_no),1)*100:.1f}% ИПУ)")
else:
    print("Колонка DEFOLT отсутствует.")

## 8. Сводка

In [ ]:
# ── Итоговая таблица ──
summary_rows = []
for seg in SEG_ORDER:
    s = df[df["segment"] == seg]
    n = len(s)
    n_sc = s["has_scenario_result"].sum()
    n_ipu = s["has_ipu"].sum()

    scr_d = scr_valid[scr_valid["segment"] == seg]["scr_duration"]
    ipu_d = ipu_valid[ipu_valid["segment"] == seg]["ipu_duration"] if len(ipu_valid) > 0 else pd.Series(dtype=float)

    summary_rows.append((
        seg,
        f"{n:,}",
        f"{s['inn'].nunique():,}",
        f"{n_sc:,} ({n_sc/n*100:.1f}%)",
        f"{n_ipu:,} ({n_ipu/n*100:.1f}%)",
        f"{n_ipu/max(n_sc,1)*100:.1f}%",
        f"{scr_d.median():.0f}" if len(scr_d) > 0 else "—",
        f"{ipu_d.median():.0f}" if len(ipu_d) > 0 else "—",
    ))

summary = pd.DataFrame(summary_rows, columns=[
    "Сегмент", "Записей", "Уник. ИНН",
    "С результатом сценария", "Перешли в ИПУ",
    "% перехода в ИПУ",
    "Медиана сценария (дни)", "Медиана ИПУ (дни)",
])

print("Сводная таблица по сегментам:")
display(summary.style.hide(axis="index"))

print("\nОсновные выводы:")
print(f"  1. Всего проанализировано {len(df):,} ФП/СФП по {df['inn'].nunique():,} компаниям.")
print(f"  2. Результат сценария заполнен у {n_with_scenario:,} ({n_with_scenario/n_total*100:.1f}%) записей.")
print(f"  3. В ИПУ перешли {n_with_ipu:,} ({n_with_ipu/n_total*100:.1f}%) записей.")
print(f"  4. Процент перехода из сценария в ИПУ: {n_with_ipu/max(n_with_scenario,1)*100:.1f}%.")